# NAA KSA Alumni Bulk Upload for Azure Static Web App

This notebook is aligned with the deployed NAA KSA web application schema.

It uploads cleaned CSV/Excel alumni rows into `AlumniProfiles` and creates/updates matching `UserAccounts` rows so users can log in immediately with:

- Username: alumni email address
- Password: alumni mobile number

Safety notes:

- Uses `PartitionKey = NUST-KSA`, matching the app.
- Uses `UserAccounts.RowKey = normalized email`, matching login.
- Preserves existing unrelated `UserAccounts` fields where possible.
- Does not seed, overwrite, or delete the three developer accounts unless the CSV contains the same email addresses.
- Does not delete duplicate profiles. It upserts by email so the same import can be safely rerun.
- Starts in validation mode. Set `APPLY_CHANGES = True` only after reviewing the preview.

In [ ]:
# Run once per Colab runtime.
!pip -q install azure-data-tables pandas openpyxl bcrypt

In [ ]:
import math
import re
import uuid
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import bcrypt
import pandas as pd
from azure.core.exceptions import ResourceExistsError, ResourceNotFoundError
from azure.data.tables import TableServiceClient, UpdateMode
from google.colab import files, userdata

UNIVERSITY_ID = "NUST-KSA"
ALUMNI_PROFILES_TABLE = "AlumniProfiles"
USER_ACCOUNTS_TABLE = "UserAccounts"

# Keep False for dry-run validation. Set True when the preview looks correct.
APPLY_CHANGES = False

VALID_ROLES = {"admin", "contributor", "alumni"}
ROLE_ALIASES = {
    "administrator": "admin",
    "admin": "admin",
    "contributor": "contributor",
    "content contributor": "contributor",
    "alumni": "alumni",
    "user": "alumni",
    "member": "alumni",
    "": "alumni",
}

# Expected Microsoft Forms export headers for the alumni onboarding form.
# Keep these in sync with the live Office Form when form questions change.
MICROSOFT_FORMS_EXPORT_FIELDS = [
    "Full Name",
    "Email Id",
    "Saudi Mobile Number",
    "Role",
    "Gender",
    "NUST Graduate Level",
    "NUST School of Graduation",
    "Degree Title",
    "Year of Graduation",
    "NUST Registration Number",
    "Visa Status in KSA",
    "Region",
    "City",
    "Current Company",
    "Current Position",
    "Industry",
    "LinkedIn Profile URL",
    "Communication Preferences",
    "Data Sharing and Privacy",
    "Reference",
]

# Minimum fields needed to create both AlumniProfiles and UserAccounts rows.
REQUIRED_IMPORT_FIELDS = ["full_name", "email", "mobile"]

COLUMN_ALIASES = {
    "full_name": ["full_name", "full name", "fullname", "name", "alumni name", "student name"],
    "email": ["email", "email id", "emailid", "email address", "e-mail", "mail", "primary email", "preferred email"],
    "mobile": ["mobile", "mobile number", "saudi mobile number", "ksa mobile number", "phone", "phone number", "contact", "contact number", "whatsapp", "whatsapp number"],
    "role": ["role", "portal role", "website role", "user role", "access role"],
    "gender": ["gender"],
    "degree": ["degree", "degree title", "program", "qualification", "nust degree", "degree program"],
    "department": ["department", "school", "nust school of graduation", "school of graduation", "faculty", "discipline", "college"],
    "graduation_year": ["graduation_year", "graduation year", "year of graduation", "grad year", "passing year", "year", "batch", "class of"],
    "current_company": ["current company", "company", "organization", "organisation", "employer", "what is the name of your organization"],
    "current_position": ["current position", "position", "designation", "job title", "current job title", "what is your current job role"],
    "industry": ["industry", "sector", "what industry are you from"],
    "linkedin_url": ["linkedin", "linkedin url", "linkedin profile", "linkedin profile url", "linkedin profile link"],
    "city": ["city", "current city", "city in ksa"],
    "country": ["country", "current country", "country of residence"],
    "region": ["region", "ksa region", "region in ksa"],
    "visa_status": ["visa status", "visa status in ksa"],
    "registration_number": ["registration number", "nust registration number"],
    "graduate_level": ["graduate level", "nust graduate level"],
    "communication_preferences": ["communication preferences", "communication preference", "preferred communication", "preferred communication channel"],
    "data_sharing_privacy": ["data sharing and privacy", "privacy", "data privacy", "privacy consent", "consent"],
    "reference": ["reference", "referred by", "referral", "comments", "notes"],
}

def utc_now_text() -> str:
    return datetime.now(timezone.utc).isoformat()

def clean_value(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and math.isnan(value):
        return ""
    try:
        if pd.isna(value):
            return ""
    except TypeError:
        pass
    if isinstance(value, float) and value.is_integer():
        return str(int(value)).strip()
    return str(value).strip()

def normalize_label(value: Any) -> str:
    text = clean_value(value).lower().replace("_", " ")
    return re.sub(r"[^a-z0-9]+", " ", text).strip()

def normalize_email(value: Any) -> str:
    return clean_value(value).lower()

def normalize_mobile(value: Any) -> str:
    text = clean_value(value)
    if text.endswith(".0") and text[:-2].isdigit():
        text = text[:-2]
    return re.sub(r"\D+", "", text)

def normalize_role(value: Any) -> str:
    role = normalize_label(value)
    return ROLE_ALIASES.get(role, "alumni")

def hash_password(password: str) -> str:
    return bcrypt.hashpw(password.encode("utf-8"), bcrypt.gensalt(rounds=12)).decode("utf-8")

def verify_password(password: str, password_hash: str) -> bool:
    if not password or not password_hash:
        return False
    return bcrypt.checkpw(password.encode("utf-8"), password_hash.encode("utf-8"))

def get_table_service_client() -> TableServiceClient:
    connection_string = userdata.get("AZURE_STORAGE_CONNECTION_STRING")
    if not connection_string:
        raise ValueError("Missing Colab secret: AZURE_STORAGE_CONNECTION_STRING")
    return TableServiceClient.from_connection_string(connection_string)

def get_table_client(table_name: str, create_if_missing: bool = True):
    service = get_table_service_client()
    client = service.get_table_client(table_name)
    if create_if_missing:
        try:
            client.create_table()
            print(f"Created table: {table_name}")
        except ResourceExistsError:
            pass
    return client

def read_source_file(file_path: str) -> pd.DataFrame:
    lower = file_path.lower()
    if lower.endswith(".csv"):
        df = pd.read_csv(file_path)
    elif lower.endswith((".xls", ".xlsx")):
        df = pd.read_excel(file_path)
    else:
        raise ValueError("Use a .csv, .xls, or .xlsx file")
    df = df.dropna(how="all").copy()
    df.columns = [clean_value(c) for c in df.columns]
    print(f"Read {len(df)} rows from {file_path}")
    return df

def detect_column_mapping(columns: List[str]) -> Dict[str, Optional[str]]:
    normalized_source = {normalize_label(column): column for column in columns}
    mapping = {}
    for target, aliases in COLUMN_ALIASES.items():
        mapping[target] = None
        for alias in aliases:
            match = normalized_source.get(normalize_label(alias))
            if match:
                mapping[target] = match
                break
    return mapping

def missing_expected_form_columns(columns: List[str]) -> List[str]:
    normalized_source = {normalize_label(column) for column in columns}
    return [field for field in MICROSOFT_FORMS_EXPORT_FIELDS if normalize_label(field) not in normalized_source]

def missing_required_import_fields(mapping: Dict[str, Optional[str]]) -> List[str]:
    return [field for field in REQUIRED_IMPORT_FIELDS if not mapping.get(field)]


## 1. Upload CSV or Excel File

Run this cell and select the cleaned Microsoft Forms export. If the file is already in Colab, comment out `files.upload()` and set `SOURCE_FILE_PATH` manually.

In [ ]:
uploaded = files.upload()
SOURCE_FILE_PATH = next(iter(uploaded.keys()))

# Example manual path:
# SOURCE_FILE_PATH = "/content/alumni.xlsx"

source_df = read_source_file(SOURCE_FILE_PATH)
column_mapping = detect_column_mapping(list(source_df.columns))

missing_form_columns = missing_expected_form_columns(list(source_df.columns))
if missing_form_columns:
    print("Warning: this export is missing expected Microsoft Forms columns:")
    for field in missing_form_columns:
        print(f"  - {field}")

missing_required = missing_required_import_fields(column_mapping)
if missing_required:
    raise ValueError(f"Missing required import fields: {missing_required}")

print("Detected mapping:")
for target, source in column_mapping.items():
    if source:
        print(f"  {target} <- {source}")
source_df.head()

## 2. Transform Rows to App-Aligned Entities

This creates both `AlumniProfiles` rows and matching `UserAccounts` rows in memory. Nothing is written yet.

In [ ]:
def row_value(row: pd.Series, target_field: str) -> str:
    source_col = column_mapping.get(target_field)
    return clean_value(row.get(source_col)) if source_col else ""

def build_profile_entity(row: pd.Series) -> Dict[str, Any]:
    email = normalize_email(row_value(row, "email"))
    if not email or "@" not in email:
        raise ValueError("missing or invalid email")

    mobile = normalize_mobile(row_value(row, "mobile"))
    full_name = row_value(row, "full_name") or email
    role = normalize_role(row_value(row, "role"))
    now = utc_now_text()

    profile = {
        "PartitionKey": UNIVERSITY_ID,
        "RowKey": email,
        "alumni_id": email,
        "full_name": full_name,
        "email": email,
        "mobile": mobile,
        "role": role,
        "status": "active",
        "visibility": "visible",
        "show_email": True,
        "show_mobile": False,
        "created_at": now,
        "updated_at": now,
    }

    optional_fields = [
        "gender", "degree", "department", "graduation_year", "current_company",
        "current_position", "industry", "linkedin_url", "city", "country", "region",
        "visa_status", "registration_number", "graduate_level", "communication_preferences",
        "data_sharing_privacy", "reference",
    ]
    for field in optional_fields:
        value = row_value(row, field)
        if value:
            profile[field] = value

    return profile

def build_user_entity(profile: Dict[str, Any], existing: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    existing = existing or {}
    email = profile["email"]
    mobile = profile.get("mobile", "")
    if not mobile:
        raise ValueError(f"missing mobile for {email}")

    password_hash = hash_password(mobile)
    if not verify_password(mobile, password_hash):
        raise ValueError(f"bcrypt verification failed for {email}")

    return {
        "PartitionKey": UNIVERSITY_ID,
        "RowKey": email,
        "user_id": clean_value(existing.get("user_id")) or str(uuid.uuid4()),
        "email": email,
        "full_name": profile.get("full_name") or existing.get("full_name") or email,
        "mobile": mobile,
        "role": normalize_role(profile.get("role")),
        "status": "approved",
        "auth_method": clean_value(existing.get("auth_method")) or "password",
        "password_hash": password_hash,
        "password_reset_required": bool(existing.get("password_reset_required", False)),
        "linked_alumni_id": profile.get("alumni_id") or email,
        "created_at": clean_value(existing.get("created_at")) or utc_now_text(),
        "updated_at": utc_now_text(),
    }

def transform_source_dataframe(df: pd.DataFrame) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], List[Dict[str, Any]]]:
    profiles_by_email: Dict[str, Dict[str, Any]] = {}
    errors: List[Dict[str, Any]] = []

    for index, row in df.iterrows():
        try:
            profile = build_profile_entity(row)
            profiles_by_email[profile["email"]] = profile
        except Exception as exc:
            errors.append({"row_number": int(index) + 2, "error": str(exc)})

    profiles = list(profiles_by_email.values())
    user_candidates = [p for p in profiles if p.get("email") and p.get("mobile")]
    return profiles, user_candidates, errors

profiles_to_upsert, users_to_sync, transform_errors = transform_source_dataframe(source_df)

preview_summary = {
    "source_rows": len(source_df),
    "unique_valid_profiles": len(profiles_to_upsert),
    "users_with_email_and_mobile": len(users_to_sync),
    "rows_with_errors": len(transform_errors),
    "roles": pd.Series([p.get("role", "alumni") for p in profiles_to_upsert]).value_counts().to_dict() if profiles_to_upsert else {},
}
preview_summary

In [ ]:
# Review invalid rows before writing.
pd.DataFrame(transform_errors).head(50)

## 3. Write to Azure Tables

Set `APPLY_CHANGES = True` in the configuration cell, then rerun from there through this cell to write data.

This cell upserts by email and does not delete any developer accounts.

In [ ]:
def get_existing_user(user_client, email: str) -> Dict[str, Any]:
    try:
        return dict(user_client.get_entity(partition_key=UNIVERSITY_ID, row_key=email))
    except ResourceNotFoundError:
        return {}

def upsert_profiles_and_users(profiles: List[Dict[str, Any]]) -> Dict[str, int]:
    if not APPLY_CHANGES:
        print("DRY RUN: no Azure Tables were changed. Set APPLY_CHANGES = True to write.")
        return {"profiles_upserted": 0, "user_accounts_upserted": 0, "user_accounts_skipped_missing_mobile": 0}

    profiles_client = get_table_client(ALUMNI_PROFILES_TABLE, create_if_missing=True)
    user_client = get_table_client(USER_ACCOUNTS_TABLE, create_if_missing=True)

    profiles_upserted = 0
    users_upserted = 0
    users_skipped = 0

    for profile in profiles:
        profile["updated_at"] = utc_now_text()
        profiles_client.upsert_entity(entity=profile, mode=UpdateMode.MERGE)
        profiles_upserted += 1

        if not profile.get("mobile"):
            users_skipped += 1
            continue

        existing_user = get_existing_user(user_client, profile["email"])
        user_entity = build_user_entity(profile, existing_user)
        user_client.upsert_entity(entity=user_entity, mode=UpdateMode.MERGE)
        users_upserted += 1

    return {
        "profiles_upserted": profiles_upserted,
        "user_accounts_upserted": users_upserted,
        "user_accounts_skipped_missing_mobile": users_skipped,
    }

write_result = upsert_profiles_and_users(profiles_to_upsert)
write_result

## 4. Verification

This reads live counts from the app partition only. It does not print password hashes.

In [ ]:
def list_partition_rows(table_name: str) -> List[Dict[str, Any]]:
    client = get_table_client(table_name, create_if_missing=False)
    return [dict(row) for row in client.query_entities(
        query_filter="PartitionKey eq @partition",
        parameters={"partition": UNIVERSITY_ID},
    )]

def verification_summary() -> Dict[str, Any]:
    profiles = list_partition_rows(ALUMNI_PROFILES_TABLE)
    users = list_partition_rows(USER_ACCOUNTS_TABLE)
    profile_emails = {normalize_email(p.get("email") or p.get("RowKey")) for p in profiles if normalize_email(p.get("email") or p.get("RowKey"))}
    user_emails = {normalize_email(u.get("email") or u.get("RowKey")) for u in users if normalize_email(u.get("email") or u.get("RowKey"))}
    missing_user_accounts = sorted(email for email in profile_emails if email not in user_emails)
    return {
        "AlumniProfiles_NUST_KSA_rows": len(profiles),
        "UserAccounts_NUST_KSA_rows": len(users),
        "profile_emails_missing_user_account": len(missing_user_accounts),
        "first_missing_user_account_emails": missing_user_accounts[:20],
        "user_roles": pd.Series([u.get("role", "") for u in users]).value_counts().to_dict() if users else {},
    }

verification_summary()

In [ ]:
# Optional bcrypt smoke test on generated in-memory sample only.
sample_password = "0501234567"
sample_hash = hash_password(sample_password)
{
    "hash_prefix_is_bcrypt_12": sample_hash.startswith("$2b$12$"),
    "verify_sample_password": verify_password(sample_password, sample_hash),
}